# Cross-Subject EEG Motor-Imagery — DL spatio-temporal CNN ensemble + per-file balanced threshold

In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
from scipy import signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.__version__)

DATA = "/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject"
TRAIN_DIR = os.path.join(DATA, "Training set")
TEST_DIR = os.path.join(DATA, "Cross participant test set")
CH = ['Fz', 'C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8', 'Oz']
FS = 250.0
WINDOW_SIZE = 500
WINDOW_STEP = 125

b_notch, a_notch = signal.iirnotch(w0=50.0, Q=30.0, fs=FS)
b_band, a_band = signal.butter(N=4, Wn=[1.0, 45.0], btype='bandpass', fs=FS)

def prep(df):
    raw = df[CH].to_numpy(np.float32)
    x = signal.filtfilt(b_notch, a_notch, raw, axis=0)
    return signal.filtfilt(b_band, a_band, x, axis=0)

def windows_from_trial(trial):
    mu = trial.mean(0, keepdims=True); sd = trial.std(0, keepdims=True)
    z = (trial - mu) / (sd + 1e-8)
    out, p = [], 0
    while p + WINDOW_SIZE <= z.shape[0]:
        out.append(z[p:p+WINDOW_SIZE, :].T.copy())
        p += WINDOW_STEP
    if not out:
        zpad = np.zeros((8, WINDOW_SIZE), np.float32)
        zpad[:, :z.shape[0]] = z.T
        out.append(zpad)
    return out

def load_dir(d, cue_vals):
    X, y, fileid, trialid = [], [], [], []
    trial_counter = 0
    for fi, f in enumerate(sorted(os.listdir(d))):
        df = pd.read_csv(os.path.join(d, f), low_memory=False, dtype={'Marker_val': str})
        m = df['Marker_val']
        trig = m.index[m.notna()].tolist()
        vals = m.iloc[trig].tolist()
        sig = prep(df)
        for i in range(len(trig) - 1):
            if vals[i] in cue_vals:
                trial = sig[trig[i]:trig[i+1], :]
                lab = 1 if vals[i] == 'move' else 0
                for w in windows_from_trial(trial):
                    X.append(w)
                    y.append(lab if cue_vals != ('cue_start',) else -1)
                    fileid.append(fi)
                    trialid.append(trial_counter)
                trial_counter += 1
    X = np.stack(X)[:, None].astype(np.float32)
    return X, np.array(y), np.array(fileid), np.array(trialid)

t0 = time.time()
Xtr, ytr, filtr, trialtr = load_dir(TRAIN_DIR, ('move', 'rest'))
print("train windows:", Xtr.shape, "trials:", int(trialtr.max())+1, round(time.time()-t0, 1), "s")
Xte, yte, filete, trialte = load_dir(TEST_DIR, ('cue_start',))
print("test windows:", Xte.shape, "trials:", int(trialte.max())+1, round(time.time()-t0, 1), "s")
import collections
ct = collections.Counter(zip(filete.tolist(), trialte.tolist()))
print("trials per test file:", {f: len(set(t for ff, t in ct if ff == f)) for f in range(int(filete.max())+1)})


In [ ]:
# ---------- architectures (MultiScaleEEGNet, ShallowConvNet, AttentionEEGNet) ----------
class MultiScaleEEGNet(nn.Module):
    def __init__(self, n_channels=8, n_classes=2, n_samples=500, F1=8, D=2, F2=16):
        super().__init__()
        self.conv_t1 = nn.Conv2d(1, F1, (1, 63), padding=(0, 31), bias=False)
        self.conv_t2 = nn.Conv2d(1, F1, (1, 31), padding=(0, 15), bias=False)
        self.conv_t3 = nn.Conv2d(1, F1, (1, 15), padding=(0, 7), bias=False)
        self.bn_t = nn.BatchNorm2d(F1 * 3)
        self.conv_s = nn.Conv2d(F1 * 3, F1 * 3 * D, (n_channels, 1), groups=F1 * 3, bias=False)
        self.bn_s = nn.BatchNorm2d(F1 * 3 * D)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.25)
        self.conv_sep_depth = nn.Conv2d(F1 * 3 * D, F1 * 3 * D, (1, 15), padding=(0, 7), groups=F1 * 3 * D, bias=False)
        self.conv_sep_point = nn.Conv2d(F1 * 3 * D, F2 * 2, (1, 1), bias=False)
        self.bn_sep = nn.BatchNorm2d(F2 * 2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(0.25)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = torch.cat([self.conv_t1(dummy), self.conv_t2(dummy), self.conv_t3(dummy)], dim=1)
            x = self.drop1(self.pool1(F.elu(self.bn_s(self.conv_s(self.bn_t(x))))))
            x = self.conv_sep_point(self.conv_sep_depth(x))
            x = self.drop2(self.pool2(F.elu(self.bn_sep(x))))
            flat_dim = x.view(1, -1).size(1)
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = torch.cat([self.conv_t1(x), self.conv_t2(x), self.conv_t3(x)], dim=1)
        x = self.bn_t(x)
        x = self.drop1(self.pool1(F.elu(self.bn_s(self.conv_s(x)))))
        x = self.conv_sep_point(self.conv_sep_depth(x))
        x = self.drop2(self.pool2(F.elu(self.bn_sep(x))))
        return self.classifier(x.view(x.size(0), -1))

class ShallowConvNet(nn.Module):
    def __init__(self, n_channels=8, n_classes=2, n_samples=500, F=40):
        super().__init__()
        self.conv_t = nn.Conv2d(1, F, (1, 25), bias=False)
        self.conv_s = nn.Conv2d(F, F, (n_channels, 1), bias=False)
        self.bn = nn.BatchNorm2d(F)
        self.pool = nn.AvgPool2d((1, 75), stride=(1, 15))
        self.drop = nn.Dropout(0.5)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.bn(self.conv_s(self.conv_t(dummy)))
            x = self.drop(torch.log(torch.clamp(self.pool(x ** 2), min=1e-6)))
            flat_dim = x.view(1, -1).size(1)
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = self.bn(self.conv_s(self.conv_t(x)))
        x = torch.log(torch.clamp(self.pool(x ** 2), min=1e-6))
        x = self.drop(x)
        return self.classifier(x.view(x.size(0), -1))

class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(1, channels // reduction), bias=False)
        self.fc2 = nn.Linear(max(1, channels // reduction), channels, bias=False)
    def forward(self, x):
        b, c, _, t = x.size()
        y = x.mean(dim=-1).view(b, c)
        y = F.relu(self.fc1(y))
        y = F.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

class AttentionEEGNet(nn.Module):
    def __init__(self, n_channels=8, n_classes=2, n_samples=500, F1=8, D=2, F2=16):
        super().__init__()
        self.conv_t = nn.Conv2d(1, F1, (1, 63), padding=(0, 31), bias=False)
        self.bn_t = nn.BatchNorm2d(F1)
        self.conv_s = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn_s = nn.BatchNorm2d(F1 * D)
        self.se1 = SqueezeExcitation(F1 * D)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.25)
        self.conv_sep_depth = nn.Conv2d(F1 * D, F1 * D, (1, 15), padding=(0, 7), groups=F1 * D, bias=False)
        self.conv_sep_point = nn.Conv2d(F1 * D, F2, (1, 1), bias=False)
        self.bn_sep = nn.BatchNorm2d(F2)
        self.se2 = SqueezeExcitation(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(0.25)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.se1(F.elu(self.bn_s(self.conv_s(self.bn_t(self.conv_t(dummy))))))
            x = self.drop1(self.pool1(x))
            x = self.se2(F.elu(self.bn_sep(self.conv_sep_point(self.conv_sep_depth(x)))))
            x = self.drop2(self.pool2(x))
            flat_dim = x.view(1, -1).size(1)
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = self.bn_t(self.conv_t(x))
        x = F.elu(self.bn_s(self.conv_s(x)))
        x = self.se1(x)
        x = self.drop1(self.pool1(x))
        x = self.conv_sep_point(self.conv_sep_depth(x))
        x = F.elu(self.bn_sep(x))
        x = self.se2(x)
        x = self.drop2(self.pool2(x))
        return self.classifier(x.view(x.size(0), -1))

print("models defined")

In [ ]:
# ---------- training engine ----------
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes=2, smoothing=0.05):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred).fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=-1))

def make_model(name, **kw):
    return {"MultiScaleEEGNet": MultiScaleEEGNet, "ShallowConvNet": ShallowConvNet,
            "AttentionEEGNet": AttentionEEGNet}[name](**kw)

def train_epochs(model, loader, opt, crit, sched, epochs, Xv, yv):
    Xvt = torch.from_numpy(Xv).to(device)
    yvt = torch.from_numpy(yv).to(device)
    best = -1.0; best_probs = None; best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = F.softmax(model(Xvt), dim=1)[:, 1].cpu().numpy()
        acc = ((vp > 0.5).astype(int) == yv).mean()
        if acc > best:
            best = acc
            best_probs = vp.copy()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return best, best_probs

def proba(model, X, bs=256):
    model.eval()
    out = []
    Xt = torch.from_numpy(X).to(device)
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            out.append(F.softmax(model(Xt[i:i+bs]), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def set_filebatch(model, on):
    """BN uses test-file batch statistics; dropout stays off."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()
    model.train() if on else model.eval()

def proba_filebatch(models, X_by_file):
    """X_by_file: list of (n_win, 1, 8, 500). Returns per-window P stacked."""
    outs = []
    for Xf in X_by_file:
        Xft = torch.from_numpy(Xf).to(device)
        ps = []
        for m in models:
            set_filebatch(m, True)
            with torch.no_grad():
                ps.append(F.softmax(m(Xft), dim=1)[:, 1].cpu().numpy())
            set_filebatch(m, False)
        outs.append(np.stack(ps, 1).mean(1))
    return np.concatenate(outs)

print("engine ready")


In [ ]:
# ---------- CV: 5-fold GroupKFold over subjects, 3 architectures ----------
EPOCHS = 18
BATCH = 64
LR = 1e-3
WD = 1e-3
N_SPLITS = 5
ARCHS = ["MultiScaleEEGNet", "ShallowConvNet", "AttentionEEGNet"]

train_files = sorted(os.listdir(TRAIN_DIR))
file_subj = [re.search(r"(S\d+)-", f).group(1) for f in train_files]
sub_of_win = np.array([file_subj[f] for f in filtr])

n_trials = int(trialtr.max()) + 1
oof_trial = np.full((n_trials, 3), np.nan)
fold_acc05, fold_accbal, fold_acc05_fb, fold_accbal_fb = [], [], [], []

gkf = GroupKFold(n_splits=N_SPLITS)
for fold, (tr_i, va_i) in enumerate(gkf.split(np.zeros(len(Xtr)), ytr, groups=sub_of_win)):
    va_trials = np.unique(trialtr[va_i])
    wva = np.isin(trialtr, va_trials)
    wtr = ~wva
    Xtr_, ytr_ = Xtr[wtr], ytr[wtr]
    Xva_ = Xtr[wva]
    yv_t = np.array([ytr[trialtr == k][0] for k in va_trials])
    ds = TensorDataset(torch.from_numpy(Xtr_), torch.from_numpy(ytr_))
    ld = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
    Xvt = torch.from_numpy(Xva_).to(device)
    final_models_keep = []
    Pw = np.zeros((len(Xva_), len(ARCHS)))
    for ai, name in enumerate(ARCHS):
        m = make_model(name).to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
        crit = LabelSmoothingLoss()
        best, best_state = -1.0, None
        for ep in range(1, EPOCHS + 1):
            m.train()
            for xb, yb in ld:
                xb = xb.to(device); yb = yb.to(device)
                opt.zero_grad()
                loss = crit(m(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
            sch.step()
            m.eval()
            with torch.no_grad():
                vp = F.softmax(m(Xvt), dim=1)[:, 1].cpu().numpy()
            acc = ((vp > 0.5).astype(int) == ytr[wva]).mean()
            if acc > best:
                best, best_state = acc, {k: v.clone() for k, v in m.state_dict().items()}
        m.load_state_dict(best_state)
        m.eval()
        with torch.no_grad():
            Pw[:, ai] = F.softmax(m(Xvt), dim=1)[:, 1].cpu().numpy()
        final_models_keep.append(m)
        del opt, sch
        torch.cuda.empty_cache()
        print(f"fold {fold} {name} best val win-acc {best:.4f}", flush=True)
    t_all = np.full(n_trials, np.nan)
    per_arch = []
    for ai in range(len(ARCHS)):
        per_arch.append(np.array([Pw[trialtr[wva] == k, ai].mean() for k in va_trials]))
    t_all[va_trials] = np.stack(per_arch, 1).mean(1)
    oof_trial[va_trials] = np.stack(per_arch, 1)
    acc05 = ((t_all[va_trials] > 0.5).astype(int) == yv_t).mean()
    def file_bal_acc(tvec):
        out = []
        for f in np.unique(filtr[wva]):
            fm = filtr[wva] == f
            tf = np.unique(trialtr[wva][fm])
            if len(tf) < 4: continue
            tfm = np.isin(va_trials, tf)
            k2 = len(tf) // 2
            P = tvec[tf]
            pred = np.zeros(len(tf), int); pred[np.argpartition(-P, k2)[:k2]] = 1
            out.append((pred == yv_t[tfm]).mean())
        return float(np.mean(out))
    accbal = file_bal_acc(t_all)
    # filebatch mode (BN per-validation-file batch stats)
    models_now = list(final_models_keep)
    Xf_list = [Xtr[wva][filtr[wva] == f] for f in np.unique(filtr[wva])]
    Pfb = proba_filebatch(models_now, Xf_list)
    t_fb = np.full(n_trials, np.nan)
    for f in np.unique(filtr[wva]):
        fm = filtr[wva] == f
        tr_w = trialtr[wva][fm]
        tf = np.unique(tr_w)
        t_fb[tf] = [Pfb[fm][tr_w == k].mean() for k in tf]
    acc05_fb = ((t_fb[va_trials] > 0.5).astype(int) == yv_t).mean()
    accbal_fb = file_bal_acc(t_fb)
    fold_acc05.append(acc05); fold_accbal.append(accbal)
    fold_acc05_fb.append(acc05_fb); fold_accbal_fb.append(accbal_fb)
    for m in models_now:
        set_filebatch(m, False)
    print(f"FOLD {fold}: eval-mode 0.5={acc05:.4f} bal={accbal:.4f} | filebatch 0.5={acc05_fb:.4f} bal={accbal_fb:.4f}", flush=True)

print("=== CV trial-level: eval-mode 0.5=%.4f bal=%.4f ===" % (np.mean(fold_acc05), np.mean(fold_accbal)))
print("=== CV trial-level: filebatch 0.5=%.4f bal=%.4f ===" % (np.mean(fold_acc05_fb), np.mean(fold_accbal_fb)))
FB_WINNER = np.mean(fold_accbal_fb) >= np.mean(fold_accbal)
print("using filebatch for submission:", FB_WINNER)
print("folds 0.5:", np.round(fold_acc05, 4))
print("folds bal:", np.round(fold_accbal, 4))
del final_models_keep
import gc; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------- dump OOF (trial-level, per-arch) for offline blend studies ----------
first_win = np.array([np.flatnonzero(trialtr == t)[0] for t in range(n_trials)])
tfile_i = filtr[first_win]
tsubj = np.array([file_subj[f] for f in tfile_i])
ty = np.array([ytr[w] for w in first_win])
oof = pd.DataFrame({"trial": np.arange(n_trials), "y": ty, "file": tfile_i, "subj": tsubj,
                    "p_ms": oof_trial[:, 0], "p_sc": oof_trial[:, 1], "p_attn": oof_trial[:, 2]})
oof.to_csv("oof_dl.csv", index=False)
print("oof dumped:", oof.shape, "meanP:", round(float(np.nanmean(oof_trial)), 4), flush=True)


In [ ]:
# ---------- final: train 3 architectures on all data, infer test ----------
ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
ld = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
final = []
for name in ARCHS:
    torch.manual_seed(SEED)
    m = make_model(name).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=16, eta_min=1e-5)
    crit = LabelSmoothingLoss()
    for ep in range(1, 17):
        m.train()
        for xb, yb in ld:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            loss = crit(m(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        sch.step()
    m.eval()
    final.append(m)
    print("trained", name, flush=True)

Xf_test = []
file_trial_idx = []
for f in range(int(filete.max()) + 1):
    fm = filete == f
    Xf_test.append(Xte[fm])
    file_trial_idx.append(np.unique(trialte[fm]))
with torch.no_grad():
    Xt = torch.from_numpy(Xte).to(device)
    Pt_eval = np.stack([F.softmax(m(Xt), dim=1)[:, 1].cpu().numpy() for m in final], 1).mean(1)
Pt_fb = proba_filebatch(final, Xf_test)
for m in final:
    set_filebatch(m, False)
nte = int(trialte.max()) + 1
tP_eval = np.zeros(nte); cnt = np.zeros(nte)
np.add.at(tP_eval, trialte, Pt_eval); np.add.at(cnt, trialte, 1)
tP_eval /= cnt
tP_fb = np.zeros(nte)
for f in range(int(filete.max()) + 1):
    tf = file_trial_idx[f]
    tr_w = trialte[filete == f]
    tP_fb[tf] = [Pt_fb[tr_w == k].mean() for k in tf]
tP = tP_fb if FB_WINNER else tP_eval
print("test trials:", nte, "| filebatch:", FB_WINNER)
for f in range(int(filete.max()) + 1):
    tf = np.unique(trialte[filete == f])
    print(f"  file {f}: n_trials={len(tf)} meanP={tP[tf].mean():.3f} move@0.5={int((tP[tf] > 0.5).sum())}")

tfile_arr = np.zeros(nte, int)
for f in range(int(filete.max()) + 1):
    tfile_arr[file_trial_idx[f]] = f
pd.DataFrame({"trial": np.arange(nte), "file": tfile_arr, "P": tP, "P_eval": tP_eval, "P_fb": tP_fb}).to_csv("test_probs.csv", index=False)
pred = np.zeros(nte, int)
ids_by_file = []
for f in range(int(filete.max()) + 1):
    tf = np.unique(trialte[filete == f])
    k2 = len(tf) // 2
    P = tP[tf]
    pred[tf[np.argpartition(-P, k2)[:k2]]] = 1
    ids_by_file.append(tf)
flat_ids = np.concatenate(ids_by_file)
assert len(flat_ids) == nte and len(np.unique(flat_ids)) == nte
rows = [{"ID": str(i), "TARGET": "move" if pred[flat_ids[i]] == 1 else "rest"} for i in range(len(flat_ids))]
dfsub = pd.DataFrame(rows)
dfsub.to_csv("submission.csv", index=False)
print(dfsub.TARGET.value_counts())
print("SUBMISSION WRITTEN rows:", len(dfsub))
df05 = pd.DataFrame([{"ID": str(i), "TARGET": "move" if tP[flat_ids[i]] > 0.5 else "rest"} for i in range(len(flat_ids))])
df05.to_csv("submission_05.csv", index=False)
print("0.5 variant moves:", int((df05.TARGET == "move").sum()))
tP_alt = tP_eval if FB_WINNER else tP_fb
pred_alt = np.zeros(nte, int)
for f in range(int(filete.max()) + 1):
    tf = file_trial_idx[f]
    k2 = len(tf) // 2
    P = tP_alt[tf]
    pred_alt[tf[np.argpartition(-P, k2)[:k2]]] = 1
pd.DataFrame([{"ID": str(i), "TARGET": "move" if pred_alt[flat_ids[i]] == 1 else "rest"} for i in range(len(flat_ids))]).to_csv("submission_alt.csv", index=False)
print("alt-mode submission written")
